In [51]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from DATA.stock_invest_function import *
from statsmodels.stats.diagnostic import acorr_ljungbox
from itertools import product
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sqlalchemy import create_engine

import matplotlib
import matplotlib.pyplot as plt
matplotlib.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
# DB 접속 정보 설정
db_info = {
    'user': 'stox7412',         # 예: 'root'
    'password': 'Apt106503!~', # 예: '1234'
    # 'host': '192.168.0.230',
    'host': 'hystox74.synology.me',         # 예: 'localhost' 또는 IP
    'port': '3307',              # 기본 포트는 보통 3306
    'database': 'investar'        # 예: 'trade_data'
}

In [4]:
trade_df = fetch_table_data(db_info, "us_trade_data")

✅ 'us_trade_data' 테이블에서 13379건의 데이터를 가져왔습니다.


In [8]:
valid_codes = trade_df['hs_code'].unique().tolist()
len(valid_codes)

96

In [13]:
def forecast_monthly_sarima_final(df, date_col='date', value_col='expDlr', steps=12, use_log=False):
    ts = df.groupby(date_col)[value_col].sum().asfreq('M')  # 월말 기준 빈도 지정

    if ts.isnull().any() or len(ts.dropna()) < 60:  # 최소 5년(60개월)
        return pd.Series(dtype='float64')

    ts_transformed = np.log(ts) if use_log else ts

    p = d = q = P = D = Q = [0, 1]
    s = 12
    param_combinations = list(product(p, d, q))
    seasonal_combinations = list(product(P, D, Q))
    total_combinations = list(product(param_combinations, seasonal_combinations))

    best_aic = np.inf
    best_model = None
    best_order = None
    best_seasonal = None

    for (order, seasonal) in total_combinations:
        seasonal_order = (*seasonal, s)
        try:
            model = SARIMAX(ts_transformed, order=order, seasonal_order=seasonal_order)
            result = model.fit(disp=False)
            if result.aic < best_aic:
                best_aic = result.aic
                best_order = order
                best_seasonal = seasonal_order
                best_model = result
        except:
            continue

    if best_model is None:
        return pd.Series(dtype='float64')

    forecast_log = best_model.forecast(steps=steps)
    forecast = np.exp(forecast_log) if use_log else forecast_log
    forecast.index = pd.date_range(start=ts.index[-1] + pd.offsets.MonthEnd(1), periods=steps, freq='M')
    return forecast


In [ ]:
quarterly_grouped['frequency'] = 'Q'

In [46]:
# SARIMA 예측 실행 (5년 이상, 로그 변환 X, 경고 제거 O)
forecast_month_list = []

for code in tqdm(valid_codes, desc="SARIMA 예측 중..."):
    trade_df['hs_code_6d'] = trade_df['hs_code'].astype(str).str[:6]
    sub_df = trade_df[trade_df['hs_code_6d'] == code].copy()
    forecast = forecast_monthly_sarima_final(sub_df[['date', 'expDlr']], steps=14, use_log=False)
    if not forecast.empty:
        temp = pd.DataFrame({
            'hs_code_6d': code,
            'date': forecast.index,
            'expDlr': forecast.values,
            'forecast': 1
        })
        forecast_month_list.append(temp)

# 기존 월별 데이터에 forecast=0
historical_df = trade_df[['hs_code_6d', 'date', 'expDlr']].copy()
historical_df['forecast'] = 0

# 전체 월별 데이터 결합
monthly_combined = pd.concat([historical_df] + forecast_month_list, ignore_index=True)
monthly_combined.sort_values(['hs_code_6d', 'date'], inplace=True)

# 분기별 집계
monthly_combined['quarter'] = monthly_combined['date'].dt.to_period('Q')
quarterly_grouped = (
    monthly_combined
    .groupby(['hs_code_6d', 'quarter'], as_index=False)['expDlr']
    .sum()
)

# 분기 말 날짜 계산
quarterly_grouped['date'] = quarterly_grouped['quarter'].dt.to_timestamp() + pd.offsets.QuarterEnd(0)

SARIMA 예측 중...: 100%|██████████| 96/96 [07:26<00:00,  4.65s/it]


In [53]:
import sqlalchemy
from sqlalchemy import create_engine

# DB 접속 정보 설정
db_info = {
    'user': 'stox7412',         # 예: 'root'
    'password': 'Apt106503!~', # 예: '1234'
    # 'host': '192.168.0.230',
    'host': 'hystox74.synology.me',         # 예: 'localhost' 또는 IP
    'port': '3307',              # 기본 포트는 보통 3306
    'database': 'investar'        # 예: 'trade_data'
}

# SQLAlchemy 엔진 생성
engine = create_engine(
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}"
)

# DataFrame 저장
quarterly_grouped.to_sql(
    name='us_trade_quarter_data_with_forecast',
    con=engine,
    if_exists='replace',  # 'append'로 하면 기존 데이터에 추가
    index=False,
    dtype={
        'hs_code_6d': sqlalchemy.types.String(length=10),
        'quarter': sqlalchemy.types.String(length=10),
        'expDlr': sqlalchemy.types.Float(),
        'date': sqlalchemy.types.Date()
    }
)

print("✅ 데이터가 성공적으로 업로드되었습니다.")

✅ 데이터가 성공적으로 업로드되었습니다.


In [55]:
# ✅ SQLAlchemy 엔진 생성
engine = create_engine(
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}"
)

# ✅ DataFrame 이름 예시 (사용자 변수명 사용)
# df = your_dataframe  # 예: us_monthly_df
monthly_combined['date'] = pd.to_datetime(monthly_combined['date'])  # 날짜 타입 보장

# ✅ DB에 저장
monthly_combined.to_sql(
    name='us_trade_monthly_data_with_forecast',
    con=engine,
    if_exists='replace',  # 기존 테이블 덮어쓰기 (append로 바꾸면 누적 저장 가능)
    index=False,
    dtype={
        'hs_code_6d': sqlalchemy.types.String(length=10),
        'date': sqlalchemy.types.Date(),
        'expDlr': sqlalchemy.types.Float(),
        'forecast': sqlalchemy.types.Integer(),
        'quarter': sqlalchemy.types.String(length=10)
    }
)

print("✅ us_trade_monthly_data_with_forecast 테이블에 데이터 저장 완료.")

✅ us_trade_monthly_data_with_forecast 테이블에 데이터 저장 완료.


In [56]:
forecast_trade_df = fetch_table_data(db_info, "us_trade_monthly_data_with_forecast")

✅ 'us_trade_monthly_data_with_forecast' 테이블에서 14639건의 데이터를 가져왔습니다.


In [60]:
forecast_trade_df[forecast_trade_df['hs_code_6d'] == '854231'].tail(12)

,hs_code_6d,date,expDlr,forecast,quarter,frequency
10707,854231,2025-07-31,3.972330e+09,1,2025Q3,M
10708,854231,2025-08-31,3.727010e+09,1,2025Q3,M
10709,854231,2025-09-30,3.457090e+09,1,2025Q3,M
10710,854231,2025-10-31,3.571190e+09,1,2025Q4,M
10711,854231,2025-11-30,3.438800e+09,1,2025Q4,M
10712,854231,2025-12-31,3.445280e+09,1,2025Q4,M
10713,854231,2026-01-31,3.668840e+09,1,2026Q1,M
10714,854231,2026-02-28,3.282410e+09,1,2026Q1,M
10715,854231,2026-03-31,3.467130e+09,1,2026Q1,M
10716,854231,2026-04-30,3.551860e+09,1,2026Q2,M
